# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [1]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os
import chromadb

C:\Users\Ousmane\AppData\Local\Temp\ipykernel_18968\4149735075.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [2]:
load_dotenv(override=True)

False

In [3]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2",temperature=0)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [4]:
pdf_file="./pdfs/Faye.babacar.CV.pdf"

In [5]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [6]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [7]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


7

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [8]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(
    model="qwen3-embedding",  # ou "qwen3-embedding:0.6b"
    dimensions=1024,  # Optionnel : ajuster selon vos besoins grâce à MRL
)
#text-embedding-3-small
#text-embedding-ada-002

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [9]:
vectorstore=Chroma.from_documents(chunks,embeddings,collection_name="cv_v2",persist_directory="./store")

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [10]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [11]:
#test
retrieved_chunks=retriever.invoke(" QU'il est la capitalisation boursière de Maroc pour 2025 ?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'total_pages': 2, 'source': './pdfs/Faye.babacar.CV.pdf', 'producer': 'Enhancv (https://www.enhancv.com)', 'title': 'Faye.babacar.CV', 'creationdate': '2026-08-13T10:47:02+00:00', 'moddate': '2026-08-13T10:47:02+00:00', 'creator': 'Enhancv', 'ecv-data': '{"_id":"69e4098b4e749040fa0b763a","style":{"background":null,"colors":["#19273C","#C4881C"],"columnLayoutOption":1,"fontBody":"interui","fontHeading":"volkhov","fontSize":3,"headerExtraSpacing":0,"isMetaDataDisabled":true,"isSectionNameUppercase":null,"layout":"ivyleague","layoutSize":"medium","lineHeightOption":0,"marginOption":0,"pageMarginOption":0,"record":"ResumeStyle","signaturePage":null},"header":{"contacts":"","customHeight":188,"dateOfBirth":"","email":"bf322003@gmail.com","extraField":"","extraLink":"https://github.com/BabacarFaye3203","height":103,"link":"https://www.linkedin.com/in/b-faye/","location":"Maroc","name":"Babacar Faye","nationality":"","phone":"+212 6 19 33 64 78","photo":"","photoStyle":"ro

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [12]:
prompt_template="""
Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Vous pouvez ajouter cette précision si vous savez que votre document concerne la Direction des études et des prévisions financières (DEPF). Vous indiquez alors le contexte suivant : *« Le contexte concerne la Note de conjoncture de la Direction des études et des prévisions financières de décembre 2025. »* C'est une phrase utile. Si vous préférez que le prompt reste général, quel que soit le fichier PDF, vous pouvez simplement retirer cette partie.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [29]:
user_input="QUel cursus scolaire a Faye Babacar ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

"BABACAR FAYE\nAI Engineer | Backend Engineer\n\x00212 6 19 33 64 78 bf322003@gmail.com https://www.linkedin.com/in/b-faye/ https://github.com/BabacarFaye3203\nMaroc\nRésumé\nIngénieur IA & Backend spécialisé en Spring Boot, IA générative et systèmes distribués. Expérience dans le développement d’API \nREST, microservices et applications basées sur les LLM avec LangChain, FastAPI, MCP et Ollama. Actuellement en Master \nIntelligence Artificielle & Digital Computing.\nCompétences\nMachine Learning Generative AI Deep learning Agentic AI Prompt Engineering RAG Fine-tuning Apache Spark Java JEE\nSpring Boot Spring Data Spring AI Spring Security API REST Microservices FastAPI Angular TypeScript React MySQL\nTDD PostgreSQL MongoDB Neo4j Hadoop Apache Kafka Docker Git GitHub Github Actions SOLID JUnit Mockito\nKubernetes Terraform\nExpérience\nZenika Casablanca, Morocco\nStage PFA \x00Software Engineer Fullstack \x00Java, DevOps & IA\x00 06/2026 - 09/2026\nMarketplace de leads et Discovery Co

In [30]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [31]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
BABACAR FAYE
AI Engineer | Backend Engineer
 212 6 19 33 64 78 bf322003@gmail.com https://www.linkedin.com/in/b-faye/ https://github.com/BabacarFaye3203
Maroc
Résumé
Ingénieur IA & Backend spécialisé en Spring Boot, IA générative et systèmes distribués. Expérience dans le développement d’API 
REST, microservices et applications basées sur les LLM avec LangChain, FastAPI, MCP et Ollama. Actuellement en Master 
Intelligence Artificielle & Digital Computing.
Compétences
Machine Learning Generative AI Deep learning Agentic AI Prompt Engineering RAG Fine-tuning Apache Spark Java JEE
Spring Boot Spring Data Spring AI Spring Security API REST Microservices FastAP

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [32]:

resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

<p>Le cursus scolaire de Faye Babacar est le suivant :</p>

<p>Il a obtenu son baccalauréat en 2024, puis a suivi des études de licence en informatique de 2024 à 2025, et enfin, il a obtenu son master en intelligence artificielle et digital computing en 2025.</p>

<p>Il a également suivi des cours de mathématiques, d'informatique, de physique et de chimie de 2022 à 2024.</p>

None


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [34]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [35]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
print(display(Markdown(resp)))

Je ne sais pas !

None


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

In [36]:
#question ou query 
user_input=" quelles certifications professionnelles a Faye Babacar ? "

Récupérer le context 

In [37]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [38]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [39]:
# Réponse d'un RAG
answer=RAG(" QU'elles sont les expériences professionnelles de Faye Babacar ? ")
print(display(Markdown(answer)))

<context>
...</context>
<question>
 QU'elles sont les expériences professionnelles de Faye Babacar ? 
</question>

La réponse est : Babacar Faye a une expérience professionnelle dans le développement d'API REST, microservices et applications basées sur les LLM avec LangChain, FastAPI, MCP et Ollama. Il a travaillé en tant que stage PFA au Zenika Casablanca, Morocco, où il a participé à la conception des API, à la modélisation des données et à l'implémentation de la logique métier dans un environnement Agile. Il a également développé un assistant basé sur un profil professionnel, connecté à Telegram pour interaction en temps réel, et une plateforme de facturation distribuée basée sur une architecture microservices event-driven.

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [40]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [41]:
# Concernant le juge, il est nécessaire d'employer un modèle plus puissant que celui mis en œuvre par le RAG. 
groundness_checker=ChatOllama(
      model="llama3.2",
      temperature=0
)

In [42]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [44]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

ResponseError: llama-server process has terminated: exit status 0xc0000409: The system detected an overrun of a stack-based buffer in this application. This overrun could potentially allow a malicious user to gain control of this application.: GGML_ASSERT(ctx->mem_buffer != NULL) failed (status code: 500)

### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
